# ❓Q Phase - Question

> Define the problem, target group, hypotheses, success metric, evaluation design and intended deliverables.
>
> _Process model reference: (Stock et al., 2020) and [docs/QUACK_process_model.md](docs/QUACK_process_model.md)_

## 1 Problem statement

Germany's electricity system is undergoing a fundamental transformation driven by the rapid expansion of renewable energy sources. In 2019, wind and solar power together accounted for approximately 35% of Germany's net electricity generation (Bundesnetzagentur & Bundeskartellamt, 2020).

Grid operators and energy planners rely on accurate short-term forecasts of electricity load to schedule generation capacity, minimise the use of costly fossil backup reserves, reduce redispatch interventions and maintain grid frequency within safe operating limits (ENTSO-E, 2023).

This project builds a data-driven, machine-learning-based forecasting pipeline for Germany's hourly electricity load, using weather data, renewable electricity generation, calendar effects and historical load patterns as input features. The project follows the QUA³CK process model and is designed to be fully reproducible from raw data to final model evaluation.

## 2 Research question and hypotheses

### 2.1 Research question

**To what extent can machine learning regression models forecast Germany's hourly electricity load for the period 2015–2019, using weather data, renewable electricity generation, calendar effects and historical lag features, achieving a symmetric mean absolute percentage error (sMAPE) of 5% or below on a time-aware held-out test set (2019)?**

> _German:_
>
> _Inwieweit können Machine-Learning-Regressionsmodelle die stündliche Stromlast Deutschlands im Zeitraum 2015–2019 auf Basis von Wetterdaten, erneuerbarer Stromerzeugung, Kalendereffekten und historischen Lag-Features mit einem symmetrischen mittleren absoluten prozentualen Fehler (sMAPE) von maximal 5% auf einem zeitlich getrennten Testdatensatz (2019) prognostizieren?_

### 2.2 Hypotheses

The following three hypotheses are derived from the research question.

1. **H1 - Calendar and seasonal effects**
    - Electricity load follows systematic patterns based on the hour of day, the day of the week and the calendar season. A model trained exclusively on calendar features will significantly outperform a 24-hour naive persistence baseline on the 2019 test set.
2. **H2 - Weather and renewable generation effects**
    - Adding weather variables (temperature, solar radiation) and renewable generation features (wind and solar output) to a calendar-only model will further reduce sMAPE by at least 10% relative, because electricity demand is temperature-dependent and renewable feed-in partially displaces dispatchable load (Hong & Fan, 2016).
3. **H3 - Historical load and lag features**
    - Lag features derived from historical electricity load (load at t−1h, t−24h, t−168h) represent the single most informative feature group and will produce the largest individual reduction in RMSE compared to any other feature group added in isolation.

**Test designs for each hypotheses:**
1. **H1:** Train a calendar-only model and compare its sMAPE to the naive baseline. H1 is rejected if the calendar model does not improve on the baseline.
2. **H2:** Compare sMAPE of calendar-only model vs. calendar + weather + renewables model on the 2019 test set. H2 is rejected if the relative improvement is below 10%.
3. **H3:** Add each feature group independently to a baseline model and compare RMSE reductions. H3 is rejected if lag features do not yield the largest individual RMSE decrease.

## 3 Target variable

| **Property**            | **Value**                                 |
| ----------------------- | ----------------------------------------- |
| **Variable name**       | `DE_load_actual_entsoe_transparency`      |
| **Source dataset**      | Open Power System Data (OPSD) Time Series |
| **Unit**                | Megawatts (MW)                            |
| **Temporal resolution** | Hourly (UTC timestamps)                   |
| **Geographic coverage** | Germany                                   |
| **Time period**         | 2015-01-01 00:00 to 2019-12-31 23:00      |
| **Modeling task**       | Supervised time-series regression         |

_The target variable represents Germany's actual hourly electricity load as reported to ENTSO-E Transparency Platform and compiled by the OPSD project (Open Power System Data, 2020)._


## 4 Target group and stakeholders

1. **Grid operators and energy analysts**
    - _Value from this project:_ Accurate hourly load forecasts support generation scheduling, reduce balancing costs and improve grid stability planning.
2. **Public infrastructure planners and policymakers**
    - _Value from this project:_ Data-driven insights into load patterns inform investment decisions for renewable capacity and grid infrastructure.
3. **Students and researchers in energy analytics**
    - _Value from this project:_ A fully documented, reproducible pipeline provides a reference implementation for load forecasting methodology.

## 5 Success metric and evaluation design

### 5.1 Primary KPI: sMAPE

The project uses **sMAPE (Symmetric Mean Absolute Percentage Error)** (Makridakis, 1993) as the single primary KPI.

$$\text{sMAPE} = \frac{1}{n} \sum_{t=1}^{n} \frac{2 \cdot |y_t - \hat{y}_t|}{|y_t| + |\hat{y}_t|} \times 100$$

**Why sMAPE as the primary KPI:**
- It is scale-independent and expressed as a percentage, making it directly interpretable: a **sMAPE of 5%** means the model is on average **95% accurate**.
- It is symmetric, penalising over- and under-forecasting equally.
- It is widely used in energy forecasting literature, making results comparable to published benchmarks.

> **Success criterion:**
> sMAPE ≤ 5 % (≥ 95 % forecasting accuracy) on the held-out 2019 test set, outperforming the 24-hour naive persistence baseline.

### 5.2 Supporting metrics

These metrics are reported alongside sMAPE for completeness and comparability, but do not drive the go/no-go decision (Hyndman & Koehler, 2006).

| **Metric** | **Role**                                                    |
| ---------- | ----------------------------------------------------------- |
| **RMSE**   | Penalises large errors during peak-load hours.              |
|            | used for H3 ablation comparison.                            |
| **MAE**    | Absolute deviation in MW. Interpretable for grid operators. |
| **R²**     | Proportion of load variance explained.                      |
|            | sanity check: target ≥ 0.95 for best model.                 |

### 5.3 Evaluation design - Time-aware split

Random train/test splits must not be used for time-series data, as they cause data leakage by allowing future information to contaminate the training set (Bergmeir & Benítez, 2012).

| **Split**      | **Period** | **Purpose**                              |
| -------------- | ---------- | ---------------------------------------- |
| **Training**   | 2015–2017  | Model learns load patterns from 3 cycles |
| **Validation** | 2018       | Hyperparameter tuning; model selection   |
| **Test**       | 2019       | Final evaluation: used exactly once      |

_The test set (2019) is held out completely and evaluated only once, after all modelling decisions have been finalised on the training and validation sets._

## 6 Scope and constraints

| **Dimension**            | **Definition**                        |
| ------------------------ | ------------------------------------- |
| **Geography**            | Germany (national level)              |
| **Time period**          | 2015–2019                             |
| **Temporal granularity** | Hourly                                |
| **Modelling task**       | Supervised time-series regression     |
| **Data inputs**          | Weather, renewable generation,        |
|                          | calendar features, lagged load values |

**Exclusions (out of scope):**
- Real-time operational deployment and live data feeds
- Dispatch optimisation or unit commitment modelling
- Sub-national or transmission-zone modelling (even though TSO-level columns exist in the raw data)
- Forecasting horizons beyond 24 hours
- Non-Germany geographies present in the raw datasets

## 7 Deployment goal and deliverables

**Deployment goal:**
A fully reproducible, publicly accessible GitHub repository containing all project artifacts, enabling any reader to re-run the complete pipeline from raw data to final model evaluation and a streamlit app.

| **Deliverable**              | **Description**                        |
| ---------------------------- | -------------------------------------- |
| **Jupyter Notebooks**        | One notebook per QUA³CK phase          |
| **Processed datasets**       | Cleaned and feature-engineered data    |
| **Model comparison results** | Visualisations of all model metrics    |
| **Visualisations**           | Time-series plots, error distributions |
| **README documentation**     | Setup, overview, results summary       |
| **Streamlit app**            | Interactive dashboard with data        |

# Bibliography

- Bergmeir, C., & Benítez, J. M. (2012). On the use of cross-validation for time
series predictor evaluation. _Information Sciences_, _191_, 192–213.
https://doi.org/10.1016/j.ins.2011.12.028
- Bundesnetzagentur & Bundeskartellamt. (2020). _Monitoringbericht 2020_.
Bundesnetzagentur. https://www.bundesnetzagentur.de
- ENTSO-E. (2023). _ENTSO-E Transparency Platform_ [Data platform].
European Network of Transmission System Operators for Electricity.
https://transparency.entsoe.eu
- Hong, T., & Fan, S. (2016). Probabilistic electric load forecasting: A tutorial
review. _International Journal of Forecasting_, _32_(3), 914–938.
https://doi.org/10.1016/j.ijforecast.2015.11.011
- Hyndman, R. J., & Koehler, A. B. (2006). Another look at measures of
forecast accuracy. _International Journal of Forecasting_, _22_(4), 679–688.
https://doi.org/10.1016/j.ijforecast.2006.03.001
- Makridakis, S. (1993). Accuracy measures: Theoretical and practical
concerns. _International Journal of Forecasting_, _9_(4), 527–529.
https://doi.org/10.1016/0169-2070(93)90079-3
- Open Power System Data. (2020). _Data package time series_ (Version 2020-10-06) [Data set]. https://doi.org/10.25832/time_series/2020-10-06
- Stock, S. C., Becker, J., Grimm, D., Hotfilter, T., Molinar, G., Stang, M., & Stork, W. (2020). _QUA³CK - A machine learning development process_. In _Proceedings of Artificial Intelligence for Science, Industry and Society — PoS(AISIS2019)_, 372, Article 026. Scuola Internazionale Superiore di Studi Avanzati. https://doi.org/10.22323/1.372.0026